# Pré-entraînement gen3 — 19 domaines de vocation, méthode EOD (ondes)

**Ce notebook exécute LA méthode d'entraînement officielle de gen3** (EOD — Entraînement par
Ondes de Domaine, it.57, `lib/neural/training/entrainement-ondes.ts`) sur les **19 domaines
de la charte de vocation** (it.59, `lib/gen3/vocation.ts`), avec :

1. les **datasets commis dans le dépôt** (11 jeux complets + 26 échantillons) — présents dans
   la session Colab après clonage ;
2. les **datasets Hugging Face publics** téléchargés avec votre token (`HF_TOKEN`) pour les
   domaines sans dataset dépôt : médecine (MedQuad), agriculture (KisanVaani), droit (legalbench) ;
3. la **charte de vocation** en repli honnête (domaine `transport-immobilier`).

**Vérification d'assimilation intégrée** : sonde de TRACE (rejeu déterministe d'échantillons
entraînés à leur horloge exacte, cortex témoin vierge contre cortex entraîné) + sonde de
généralisation (textes jamais entraînés, résultat rapporté tel quel).

Aucun service d'IA externe : le SNN est **purement local** (cortex columnnaire 1 048 576
colonnes simulé en TypedArrays — le GPU n'est pas nécessaire, un runtime CPU suffit).

**Durée réelle mesurée : quelques secondes.**

In [ ]:
# 1. Environnement : Node.js 20 (pas besoin de GPU — l'EOD est un substrat CPU événementiel)
!nvidia-smi 2>/dev/null | head -4 || echo "Pas de GPU alloué — sans importance : la méthode EOD est événementielle (coût ∝ activité), elle tourne sur CPU."
!node --version 2>/dev/null || (curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && sudo apt-get install -y nodejs)
!echo "node: $(node --version) — npm: $(npm --version)"

In [ ]:
# 2. Clonage du dépôt (token GitHub demandé en saisie MASQUÉE, jamais écrit sur disque)
import os, getpass
if not os.path.exists('/content/gen3ia/.git'):
    pat = os.environ.get('GEN3IA_PAT') or getpass.getpass('Token GitHub (ghp_…) : ')
    os.environ['GEN3IA_PAT'] = pat
    !git clone --depth 1 https://$GEN3IA_PAT@github.com/missock237-spec/gen3ia /content/gen3ia
%cd /content/gen3ia
!git log --oneline -1
!echo "Datasets présents dans la session Colab : $(ls datasets/ | wc -l) dossiers"
!ls datasets/ | head -8 && echo "…"

In [ ]:
# 3. PRÉ-ENTRAÎNEMENT EOD — les 19 domaines (token HF demandé en saisie MASQUÉE)
import os, getpass
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN') or getpass.getpass('Token Hugging Face (hf_…) : ')
os.environ['GEN3IA_ENV'] = 'google-colab'
!npx -y tsx@4.19.2 scripts/preentrainement-colab.ts

In [ ]:
# 4. Vérification lisible : gen3 a-t-il assimilé les domaines ?
import json
r = json.load(open('assets/neural-pretrained/preentrainement-eod-19-domaines.json'))
print(f"Méthode   : {r['methode']}")
print(f"Environn. : {r['environnement']['type']} — token HF fourni : {r['environnement']['hfTokenFourni']}")
print(f"Sources   : dépôt {r['sources']['depot']} + Hugging Face {r['sources']['huggingface']} + charte {r['sources']['charte']}")
print(f"Session   : {r['totalEchantillons']} échantillons en {r['dureeMs']/1000:.1f} s")
print()
print("ASSIMILATION (sonde de trace : gain du ratio porteur/autres, témoin vierge → entraîné) :")
for d in r['sonde']['domaines']:
    t = d['trace']
    print(f"  {d['domaine']:<30} {t['avant']:.2f} → {t['apres']:.2f}  gain ×{t['gain']:.2f}  {'✓ assimilé' if t['assimile'] else '✗ NON'}")
print()
print(f"Bilan : {r['sonde']['assimiles']} / {r['sonde']['total']} domaines assimilés (trace ondulaire mesurée).")
m = r['metabolisme']
print(f"Métabolisme réel : {m['evaluees']:,} neurones évalués, {m['reposes']:,} au repos (coût zéro), {m['ms']} ms.".replace(',', ' '))
print()
print("Le rapport complet est dans assets/neural-pretrained/preentrainement-eod-19-domaines.json")
print("(téléchargeable depuis l'onglet Fichiers de Colab) — la sonde de généralisation y est")
print("rapportée TELLE QUELLE : les textes jamais entraînés ne montrent pas de gain, conforme")
print("aux « Limites honnêtes » de la méthode (le balayage ondulaire étale la trace, la")
print("généralisation passe par les échantillons, pas par un plongement pré-entraîné).")